# Delta Lake Operations Examples

This notebook covers common Delta Lake patterns used in Databricks:

- creating a Delta table
- merge for upserts
- update and delete operations
- time travel queries
- table history inspection

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [ ]:
target_table = "main.demo.delta_customer_status"

base_data = [
    (1, "Asha", "active", 1200.0),
    (2, "Miguel", "active", 900.0),
    (3, "Noah", "inactive", 100.0)
]

base_df = spark.createDataFrame(base_data, ["customer_id", "customer_name", "status", "spend"])

base_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
print(f"Created or replaced {target_table}")

In [ ]:
display(spark.table(target_table).orderBy("customer_id"))

## Merge example

Use merge when you need to upsert records into an existing Delta table.

In [ ]:
changes_data = [
    (2, "Miguel", "active", 1100.0),
    (3, "Noah", "active", 250.0),
    (4, "Lina", "active", 500.0)
]

changes_df = spark.createDataFrame(changes_data, ["customer_id", "customer_name", "status", "spend"])
delta_table = DeltaTable.forName(spark, target_table)

(
    delta_table.alias("target")
    .merge(changes_df.alias("source"), "target.customer_id = source.customer_id")
    .whenMatchedUpdate(set={
        "customer_name": "source.customer_name",
        "status": "source.status",
        "spend": "source.spend"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "status": "source.status",
        "spend": "source.spend"
    })
    .execute()
)

In [ ]:
display(spark.table(target_table).orderBy("customer_id"))

## Update and delete examples

Delta Lake supports row-level modifications directly on the table.

In [ ]:
delta_table.update(
    condition="customer_id = 4",
    set={"spend": "750.0", "status": "active"}
)

delta_table.delete(condition="customer_id = 1 AND spend < 1500")

In [ ]:
display(spark.table(target_table).orderBy("customer_id"))

## History and time travel

Every write creates a new Delta version, which you can inspect and query.

In [ ]:
history_df = spark.sql(f"DESCRIBE HISTORY {target_table}")
display(history_df)

In [ ]:
version_zero_df = spark.sql(f"SELECT * FROM {target_table} VERSION AS OF 0")
display(version_zero_df.orderBy("customer_id"))

## Production notes

- Use merge for CDC and incremental upserts
- Use table history to debug when data changed
- Use time travel carefully for audit and validation use cases
- Place production tables in governed Unity Catalog schemas